# Biomolecular simulation with GROMACS

In this tutorial, we will go over the very basics of standard biomolecular simulation. For that, we will use [GROMACS](https://www.gromacs.org/), one of the most widely used MD engines when it comes to biomolecules. GROMACS is a highly optimised package, with literal decades of development and the first choice for thousands of researches around the world. This tutorial will NOT be an exhaustive look into everything one can do with GROMACS, since the ecosystem is huge and can take years to master completely. Instead, we will familiarise ourselves with its basic command line interface and, hopefully, in the end we will be able to comfortably set up simple systems and run unrestrained MD simulations.

## Part 1 - Preparing a protein simulation

In the first part of the tutorial we will learn how to prepare a small protein-in-water system and get it ready for a production simulation. We will start from a PDB file containing only the protein and end up with some GROMACS binaries ready to run.

### 1.1 - Familiarising ourselves with the system

The very first thing we should really do is visualize the system that we want to simulate. 

The following display is interactive, play with it for a while. What kind of molecule are we dealing with? What can you tell me about it? Why does it "move" if it is only one PDB file?

In [4]:
import MDAnalysis as mda
import nglview as nv

u_2RVD = mda.Universe("structures/2RVD.pdb")
nv.show_mdanalysis(u_2RVD)

NGLWidget(max_frame=19)

It is probably a good idea we inspect the PDB file. Open it in another tab. You will see ato some point these two lines:

```text
KEYWDS    BETA-HAIRPIN, MINI-PROTEIN, CHIGNOLIN, DE NOVO PROTEIN                
EXPDTA    SOLUTION NMR
```

Here we actually have some answers to the previous questions! We are dealing with a *de novo* designed protein, a beta-hairpin, studied through solution NMR. A bit further down we will read:

```text
REMARK 210 BEST REPRESENTATIVE CONFORMER IN THIS ENSEMBLE : 1
```

So this is a conformational ensemble! That explains why the molecule "moves" in the visualiser, every frame corresponds to one conformation. We are also told that the most representative conformer is the first one. Probably, then, it is a good idea if we save it.

In [5]:
u_2RVD.trajectory[0] # Conformers are stored as if they were a trajectory
u_2RVD.atoms.write("exercise-1/2RVD_conf1.pdb")

u_conf1 = mda.Universe("exercise-1/2RVD_conf1.pdb")
nv.show_mdanalysis(u_conf1)


/lmb/home/alexandrebg/.local/share/mamba/envs/gromacs-tutorial/lib/python3.12/site-packages/MDAnalysis/coordinates/PDB.py:885: UserWarning: Unit cell dimensions not found. CRYST1 record set to unitary values.
  warnings.warn(
/lmb/home/alexandrebg/.local/share/mamba/envs/gromacs-tutorial/lib/python3.12/site-packages/MDAnalysis/coordinates/PDB.py:1282: UserWarning: Found no information for attr: 'formalcharges' Using default value of '0'
  warnings.warn(
/lmb/home/alexandrebg/.local/share/mamba/envs/gromacs-tutorial/lib/python3.12/site-packages/MDAnalysis/coordinates/PDB.py:479: UserWarning: 1 A^3 CRYST1 record, this is usually a placeholder. Unit cell dimensions will be set to None.
  warnings.warn(


NGLWidget()

Now we can see that there is no play button on the visualizer, and have just one structure. This will be our starting point and we are now ready to really start thinking about setting up the simulation.

### 1.2 - Preparation of the structure

Let's think now about where does this protein live. Is it in vaccuum? Does it occupy any space at all? Surely, it must, right? Luckily PDB files provide us with that kind of information. If we keep reading the PDB file we will bump into these lines:

```text
REMARK 215 THE COORDINATES IN THIS ENTRY WERE GENERATED FROM SOLUTION           
REMARK 215 NMR DATA.  PROTEIN DATA BANK CONVENTIONS REQUIRE THAT                
REMARK 215 CRYST1 AND SCALE RECORDS BE INCLUDED, BUT THE VALUES ON              
REMARK 215 THESE RECORDS ARE MEANINGLESS.  
```

What this means, really, is that there is really no unit-cell information for this system, this is, for all effects the protein is sitting in vacuum in an infinitely large box. That is not desirable for a condensed matter system. We would like our protein to be contained in a finite volume and be solvated by something.

Let's start with putting our protein in a finite box. This will be the first GROMACS command we will run. Open a terminal, and type the following

```bash
gmx editconf -f exercise-1/2RVD_conf1.pdb -o exercise-1/2RVD_box.pdb -box 5 5 5
```

`editconf` means edit conformation, and is a command that allows simple manipulations of structure files, like translations, rotations and, in this case, adding a simulation box. We use `-f` to indicate the input file, `-o` for an output file and `-box` to give the size of the box in nanometres. By default, the box is an ortorrhombic box, so the angles between the box vectors will be 90 degrees. `editconf` can take many more options, you can inspect all you can do with it if you run `gmx editconf -h`; and this is true for all available GROMACS commands.

We can now visualise the result of the previous operation.

In [9]:
u_box = mda.Universe("exercise-1/2RVD_box.pdb")

view = nv.show_mdanalysis(u_box)
view.add_cartoon(selection="protein")
view.add_ball_and_stick(selection="protein")
view.add_unitcell()
view.center()
view

NGLWidget()

We now see a box around our protein! Now GROMACS will understand that the protein must live at all times in that volume and does not have (in principle) an infinite space to go around. We will see in a bit why I said *in principle*. 

However, our protein still looks quite lonely in the middle of that box. In nature, proteins usually do not appear in vacuum, unless something goes really, *really* wrong. We must therefore fix this and **solvate** the protein. The solvent of choice will be, of course, water; since that is the solvent of life. We can use GROMACS for that, in the same terminal you opened before, run:

```bash
gmx solvate -cp exercise-1/2RVD_box.pdb -o exercise-1/2RVD_solv.pdb
```

In this case `solvate` is pretty much self-explanatory. `-cp` stands for Conformation of the Protein and is used to pass the conformation of the thing to be solvated. Despite its name, you do NOT need to pass necessarily a protein, it can really be anything. Again, with `-o` we decide the output file.

We can visualise the result of this:

In [14]:
u_solv = mda.Universe("exercise-1/2RVD_solv.pdb")

view = nv.show_mdanalysis(u_solv)
view.clear_representations()
view.add_cartoon(selection="protein")
view.add_ball_and_stick(selection="resname SOL")
view.add_unitcell()
view

/lmb/home/alexandrebg/.local/share/mamba/envs/gromacs-tutorial/lib/python3.12/site-packages/MDAnalysis/topology/PDBParser.py:372: UserWarning: Unknown element  found for some atoms. These have been given an empty element record. If needed they can be guessed using universe.guess_TopologyAttrs(context='default', to_guess=['elements']).
  warnings.warn(wmsg)


NGLWidget()

Now, that looks much better! We have quite a few waters accompanying our protein now. However, you may notice something that could seem strange at first: why are some water molecules **outside** the simulation box?

The answer to that question is that they actually aren't! In condensed-matter simulation we will work, more often than not, under Periodic Boundary Conditions (PBC). Think about pac-man, he comes out one side of the map and immediately appears again through the wall in the opposite side. This is exactly what is happening here, the water molecules that appear outside the box are actually inside, just on the opposite side.

With this, we have now a much more decent system and we can move to the next part of the tutorial.

### 1.3 - Energy minimisation

So far, we 